In [69]:
import pandas as pd
import requests

In [70]:
url = "https://api.nobelprize.org/2.1/laureates"

headers = {
    "User-Agent": "ProjetoAnaliseNobel (contato: fdias110108@gmail.com)",
    "Accept": "application/json"
}

In [71]:

response = requests.get(
    url,
    params={"limit": 1029},
    headers=headers,
    timeout=30
)
response.raise_for_status()
dados = response.json()

df = pd.json_normalize(dados["laureates"])


In [74]:
colunas = [
    "id",
    "fullName.en",
    "orgName.en",
    "gender",
    "birth.date",
    "birth.year",
    "birth.place.city.en",
    "birth.place.country.en",
    "birth.place.countryNow.en",
    "birth.place.continent.en",
    "death.date",
    "founded.date",
    "foundedCountryNow.en",
    "wikidata.id",
    "wikipedia.english",
    "nobelPrizes"
]

In [75]:
df_nobel = df.reindex(columns = colunas).copy()

In [77]:
df_nobel = df_nobel.rename(columns = {
    "fullName.en": "nome",
    "orgName.en": "organizacao",
    "gender": "genero",
    "birth.date": "nascimento",
    "birth.year": "ano_nascimento",
    "birth.place.city.en": "cidade",
    "birth.place.country.en": "pais_nascimento",
    "birth.place.countryNow.en": "pais_atual",
    "birth.place.continent.en": "continente",
    "death.date": "falecimento",
    "founded.date": "fundacao",
    "foundedCountryNow.en": "pais_organizacao",
    "wikidata.id": "wikidata_id",
    "wikipedia.english": "wikipedia",
    "nobelPrizes": "premios"
})

In [78]:
df_nobel["nome"] = df_nobel["nome"].fillna(df_nobel["organizacao"])
df_nobel = df_nobel.drop(columns = ["organizacao"])

In [82]:
df_nobel["tipo"] = "Pessoa"

df_nobel.loc[
    df_nobel["genero"].isna(),
    "tipo"
] = "Organização"

In [ ]:
df_premios = df_nobel.explode("premios", ignore_index = True).copy()

,nome,premios
0,A. Michael Spence,"{'awardYear': '2001', 'category': {'en': 'Econ..."
1,Aage Niels Bohr,"{'awardYear': '1975', 'category': {'en': 'Phys..."
2,Aaron Ciechanover,"{'awardYear': '2004', 'category': {'en': 'Chem..."
3,Aaron Klug,"{'awardYear': '1982', 'category': {'en': 'Chem..."
4,Abdulrazak Gurnah,"{'awardYear': '2021', 'category': {'en': 'Lite..."
5,Abdus Salam,"{'awardYear': '1979', 'category': {'en': 'Phys..."
6,Abhijit Banerjee,"{'awardYear': '2019', 'category': {'en': 'Econ..."
7,Abiy Ahmed Ali,"{'awardYear': '2019', 'category': {'en': 'Peac..."
8,Ada E. Yonath,"{'awardYear': '2009', 'category': {'en': 'Chem..."
9,Adam G. Riess,"{'awardYear': '2011', 'category': {'en': 'Phys..."


In [84]:
dados_premios = pd.json_normalize(df_premios["premios"].tolist())

In [85]:
colunas_premios = [
    "awardYear",
    "category.en",
    "motivation.en",
    "portion"
]
dados_premios = dados_premios.reindex(
    columns=colunas_premios
)
dados_premios.columns = [
    "ano_premio",
    "categoria",
    "motivacao",
    "parcela_premio"
]

In [86]:
df_premios = pd.concat(
    [
        df_premios.drop(columns=["premios"]),
        dados_premios
    ],
    axis=1
)

In [87]:
df_premios["ano_premio"] = pd.to_numeric(
    df_premios["ano_premio"],
    errors = "coerce"
).astype("Int64")
df_premios["ano_nascimento"] = pd.to_numeric(
    df_premios["ano_nascimento"],
    errors="coerce"
).astype("Int64")
df_premios["idadde_aproximada"] = (df_premios["ano_premio"]-df_premios["ano_nascimento"])

In [88]:
df_premios.loc[
    df_premios["tipo"] == "Organização",
    "idade_aproximada"
] = pd.NA

In [89]:
print([coluna for coluna in df.columns
       if "founded" in coluna.lower()])

['founded.date', 'founded.place.city.en', 'founded.place.city.no', 'founded.place.city.se', 'founded.place.country.en', 'founded.place.country.no', 'founded.place.country.se', 'founded.place.country.sameAs', 'founded.place.cityNow.en', 'founded.place.cityNow.no', 'founded.place.cityNow.se', 'founded.place.cityNow.sameAs', 'founded.place.countryNow.en', 'founded.place.countryNow.no', 'founded.place.countryNow.se', 'founded.place.countryNow.sameAs', 'founded.place.continent.en', 'founded.place.continent.no', 'founded.place.continent.se', 'founded.place.locationString.en', 'founded.place.locationString.no', 'founded.place.locationString.se', 'foundedCountry.en', 'foundedCountry.no', 'foundedCountry.se', 'foundedCountryNow.en', 'foundedCountryNow.no', 'foundedCountryNow.se', 'foundedContinent.en']


In [90]:
df_nobel["ano_nascimento"] = pd.to_numeric(
    df_nobel["ano_nascimento"],
    errors="coerce"
).astype("Int64")

In [91]:
colunas_datas = [
    "nascimento",
    "falecimento",
    "fundacao"
]

for coluna in colunas_datas:
    df_nobel[coluna] = pd.to_datetime(
        df_nobel[coluna],
        errors="coerce",
        format="mixed"
    )

In [92]:
print(df_nobel[colunas_datas])

     nascimento falecimento fundacao
0    1943-11-07         NaT      NaT
1    1922-06-19  2009-09-08      NaT
2    1947-10-01         NaT      NaT
3    1926-08-11  2018-11-20      NaT
4           NaT         NaT      NaT
...         ...         ...      ...
1023 1921-01-18  2015-07-05      NaT
1024 1945-02-09         NaT      NaT
1025 1936-11-19         NaT      NaT
1026 1930-10-10  2015-01-27      NaT
1027 1930-03-15  2019-03-01      NaT

[1028 rows x 3 columns]
